In [1]:
# C:/IDEAL_Programming/src/train_lgbm_api_generic_models.py
# ============================================================
# FINAL GENERIC UI LIGHTGBM API MODELS TRAINING SCRIPT
# ============================================================
#
# Purpose
# -------
# Trains LightGBM models suitable for the IDEAL API/UI where the
# end user does NOT provide home_id.
#
# Models trained:
#
# 1) WITH-HISTORY GENERIC MODEL
#    - Uses lag / rolling consumption features
#    - Uses static, temporal and weather/environmental features
#    - DOES NOT use home_id as a model feature
#    - DOES NOT use train-only home statistics
#    - DOES NOT use consumption_regime derived from known home_id
#    - Suitable for generic UI users who provide recent consumption history
#
# 2) no-history LIGHTGBM MODEL
#    - Uses static + temporal + weather/environmental features
#    - Uses train-only behavioral static-group profiles
#    - Uses KNN similar-home profiles from static attributes
#    - DOES NOT use home_id as a model feature
#    - Suitable as optional comparison model in the UI
#    - The final no-history UI default can still be Random Forest
#
# Output:
#   C:/IDEAL_Programming/processed/models/final_api_models/LGBM_generic_ui/
#     ├── training_summary.json
#     ├── with_history_generic/
#     └── no_history_optional/
#
# Important UI note
# -----------------
# home_id is used only internally during training/evaluation to compute lag
# features per measured home and to evaluate metrics per home-day. It is not
# included in the model feature matrix for the generic UI models.
# ============================================================

from __future__ import annotations

import json
import math
import warnings
from pathlib import Path
from typing import Any, Dict, Tuple

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

BASE_DIR = Path("C:/IDEAL_Programming")
DATA_PATH = BASE_DIR / "processed" / "final" / "IDEAL_final_hourly_dataset.csv"

OUT_ROOT = BASE_DIR / "processed" / "models" / "final_api_models" / "LGBM2"
WITH_HISTORY_DIR = OUT_ROOT / "with_history_generic"
no_history_DIR = OUT_ROOT / "no_history_optional"

OUT_ROOT.mkdir(parents=True, exist_ok=True)
WITH_HISTORY_DIR.mkdir(parents=True, exist_ok=True)
no_history_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

RANDOM_STATE = 42

TEST_DAYS = 30
VAL_DAYS_TOTAL = 30
CAL_DAYS = 15

BAD_HOMES = ["home171", "home219", "home228", "home271"]

USE_LOG_TARGET = True
CLIP_TARGET_TRAIN = False

TRAIN_WITH_HISTORY_GENERIC = True
TRAIN_no_history_OPTIONAL = True

# no-history KNN settings
N_KNN_NEIGHBORS = 15
REMOVE_SAME_HOME_FROM_KNN = True

MIN_STATIC_DOW_HOUR_COUNT = 50
MIN_STATIC_HOUR_COUNT = 100
MIN_STATIC_OVERALL_COUNT = 200

# Features that can be supplied or imputed in the API/UI
STATIC_NUMERIC = [
    "external_temperature",
    "internal_temperature",
    "residents",
    "total_floor_area_m2",
    "num_electric_appliances",
]

TIME_NUMERIC = [
    "hour",
    "day_of_week",
    "month",
    "day_of_month",
    "week_of_year",
    "is_weekend",
    "is_holiday",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos",
]

BASE_CATEGORICAL = [
    "hometype",
    "urban_rural_class",
]

# With-history generic config
# Keep maximum required history at 168 hours for practical UI usage.
LAG_HOURS = [1, 2, 3, 6, 12, 24, 48, 72, 168]
ROLLING_WINDOWS = [3, 6, 12, 24, 48, 168]
ROLLING_EXTREME_WINDOWS = [24, 48, 168]

REQUIRED_HISTORY_COLS = [
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "roll_mean_24h",
    "roll_mean_168h",
]

MIN_REQUIRED_HISTORY_HOURS = 168

HISTORY_LGBM_PARAMS = dict(
    n_estimators=9000,
    learning_rate=0.012,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=80,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.05,
    reg_lambda=0.20,
    objective="regression",
    random_state=RANDOM_STATE + 5,
    n_jobs=-1,
    force_row_wise=True,
    verbosity=-1,
)

# no-history config
NOHISTORY_EXTRA_NUMERIC = [
    "expected_behavior_Wh",
    "expected_behavior_log1p",
    "knn_expected_dow_hour_Wh",
    "knn_expected_hour_Wh",
    "knn_expected_overall_Wh",
    "knn_expected_Wh",
    "knn_expected_log1p",
    "knn_minus_behavior_Wh",
]

NOHISTORY_CATEGORICAL = [
    "hometype",
    "urban_rural_class",
    "residents_bin",
    "area_bin",
    "appliances_bin",
    "behavior_profile_source",
]

NOHISTORY_LGBM_PARAMS = dict(
    n_estimators=7000,
    learning_rate=0.015,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=80,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.05,
    reg_lambda=0.20,
    objective="regression",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    force_row_wise=True,
    verbosity=-1,
)


# ============================================================
# JSON / METRICS UTILS
# ============================================================

def to_jsonable(obj: Any) -> Any:
    """Convert numpy/pandas objects to JSON-safe Python objects."""
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if pd.isna(obj) if isinstance(obj, (float, np.floating)) else False:
        return None
    return obj


def save_json(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_jsonable(obj), f, ensure_ascii=False, indent=2)


def mape_safe(y_true, y_pred, eps: float = 1e-6) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mask = y_true > eps
    if mask.sum() == 0:
        return float("nan")
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def smape_safe(y_true, y_pred, eps: float = 1e-6) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100)


def compute_metrics(y_true, y_pred) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(math.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE_%": mape_safe(y_true, y_pred),
        "sMAPE_%": smape_safe(y_true, y_pred),
        "bias_Wh": float(np.mean(y_pred - y_true)),
    }


def evaluate_predictions(df_eval: pd.DataFrame, pred_col: str) -> Dict[str, float]:
    y_true = df_eval[TARGET_COL].values
    y_pred = df_eval[pred_col].values

    metrics = compute_metrics(y_true, y_pred)

    daily = (
        df_eval
        .assign(date=df_eval[TIME_COL].dt.date)
        .groupby([HOME_COL, "date"], as_index=False)
        .agg(
            actual_kWh=(TARGET_COL, lambda x: x.sum() / 1000),
            pred_kWh=(pred_col, lambda x: x.sum() / 1000),
        )
    )

    daily["abs_error_kWh"] = np.abs(daily["actual_kWh"] - daily["pred_kWh"])
    daily["abs_error_pct"] = daily["abs_error_kWh"] / daily["actual_kWh"].replace(0, np.nan) * 100

    metrics.update({
        "daily_abs_error_kWh_mean": float(daily["abs_error_kWh"].mean()),
        "daily_abs_error_pct_mean": float(daily["abs_error_pct"].mean()),
        "num_home_days": int(len(daily)),
    })

    return metrics


def print_metrics(title: str, metrics: Dict[str, float]) -> None:
    print("=" * 100)
    print(title)
    print("=" * 100)
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")


def transform_target(y):
    y = np.asarray(y, dtype=np.float64)
    if USE_LOG_TARGET:
        return np.log1p(np.maximum(y, 0.0))
    return y


def inverse_target(y_hat):
    y_hat = np.asarray(y_hat, dtype=np.float64)
    if USE_LOG_TARGET:
        return np.expm1(y_hat)
    return y_hat


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def build_preprocessor(numeric_cols, categorical_cols):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
            ("onehot", make_one_hot_encoder()),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


# ============================================================
# COMMON FEATURE ENGINEERING
# ============================================================

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df[TIME_COL] = pd.to_datetime(df[TIME_COL])
    df["hour"] = df[TIME_COL].dt.hour
    df["day_of_week"] = df[TIME_COL].dt.dayofweek
    df["month"] = df[TIME_COL].dt.month
    df["day_of_month"] = df[TIME_COL].dt.day
    df["week_of_year"] = df[TIME_COL].dt.isocalendar().week.astype(int)
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

    if "is_holiday" not in df.columns:
        df["is_holiday"] = 0

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    return df


def add_static_bins(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for c in ["hometype", "urban_rural_class"]:
        if c not in df.columns:
            df[c] = "unknown"
        df[c] = df[c].fillna("unknown").astype(str).replace({"nan": "unknown", "None": "unknown"})

    if "residents" not in df.columns:
        df["residents"] = np.nan
    if "total_floor_area_m2" not in df.columns:
        df["total_floor_area_m2"] = np.nan
    if "num_electric_appliances" not in df.columns:
        df["num_electric_appliances"] = np.nan

    r = pd.to_numeric(df["residents"], errors="coerce")
    df["residents_bin"] = np.select(
        [r.isna(), r <= 0, r == 1, r == 2, r == 3, r == 4, r >= 5],
        ["unknown", "unknown", "1", "2", "3", "4", "5plus"],
        default="unknown",
    )

    area = pd.to_numeric(df["total_floor_area_m2"], errors="coerce")
    df["area_bin"] = pd.cut(
        area,
        bins=[-np.inf, 50, 80, 110, 150, np.inf],
        labels=["area_0_50", "area_50_80", "area_80_110", "area_110_150", "area_150plus"],
    ).astype(str)
    df["area_bin"] = df["area_bin"].replace({"nan": "unknown", "None": "unknown"})

    app = pd.to_numeric(df["num_electric_appliances"], errors="coerce")
    df["appliances_bin"] = pd.cut(
        app,
        bins=[-np.inf, 5, 10, 20, 30, np.inf],
        labels=["app_0_5", "app_5_10", "app_10_20", "app_20_30", "app_30plus"],
    ).astype(str)
    df["appliances_bin"] = df["appliances_bin"].replace({"nan": "unknown", "None": "unknown"})

    df["static_profile_key"] = (
        df["hometype"].astype(str) + "|" +
        df["urban_rural_class"].astype(str) + "|" +
        df["residents_bin"].astype(str) + "|" +
        df["area_bin"].astype(str) + "|" +
        df["appliances_bin"].astype(str)
    )

    return df


def temporal_split(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    max_time = df[TIME_COL].max()

    test_start = max_time - pd.Timedelta(days=TEST_DAYS) + pd.Timedelta(hours=1)
    val_start = test_start - pd.Timedelta(days=VAL_DAYS_TOTAL)
    cal_start = test_start - pd.Timedelta(days=CAL_DAYS)

    train_df = df[df[TIME_COL] < val_start].copy()
    es_df = df[(df[TIME_COL] >= val_start) & (df[TIME_COL] < cal_start)].copy()
    cal_df = df[(df[TIME_COL] >= cal_start) & (df[TIME_COL] < test_start)].copy()
    test_df = df[df[TIME_COL] >= test_start].copy()

    return train_df, es_df, cal_df, test_df


# ============================================================
# WITH-HISTORY GENERIC FEATURES
# ============================================================

def add_lag_and_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create lag and rolling features per home.

    home_id is used only for grouping during training data preparation.
    It is not used as a model input feature.
    """
    df = df.sort_values([HOME_COL, TIME_COL]).copy()

    g = df.groupby(HOME_COL)[TARGET_COL]

    for lag in LAG_HOURS:
        df[f"lag_{lag}h"] = g.shift(lag)

    shifted = g.shift(1)

    for w in ROLLING_WINDOWS:
        df[f"roll_mean_{w}h"] = (
            shifted
            .groupby(df[HOME_COL])
            .rolling(w, min_periods=max(2, w // 4))
            .mean()
            .reset_index(level=0, drop=True)
        )

    for w in ROLLING_EXTREME_WINDOWS:
        df[f"roll_std_{w}h"] = (
            shifted
            .groupby(df[HOME_COL])
            .rolling(w, min_periods=max(2, w // 4))
            .std()
            .reset_index(level=0, drop=True)
        )

        df[f"roll_min_{w}h"] = (
            shifted
            .groupby(df[HOME_COL])
            .rolling(w, min_periods=max(2, w // 4))
            .min()
            .reset_index(level=0, drop=True)
        )

        df[f"roll_max_{w}h"] = (
            shifted
            .groupby(df[HOME_COL])
            .rolling(w, min_periods=max(2, w // 4))
            .max()
            .reset_index(level=0, drop=True)
        )

    df["lag_1h_minus_24h"] = df["lag_1h"] - df["lag_24h"]
    df["lag_24h_minus_168h"] = df["lag_24h"] - df["lag_168h"]
    df["lag_48h_minus_168h"] = df["lag_48h"] - df["lag_168h"]
    df["roll_24h_div_168h"] = df["roll_mean_24h"] / (df["roll_mean_168h"] + 1.0)
    df["roll_3h_div_24h"] = df["roll_mean_3h"] / (df["roll_mean_24h"] + 1.0)

    return df


def fit_global_daily_calibrator(cal_eval: pd.DataFrame, pred_col: str) -> Dict[str, Any]:
    """Optional generic daily calibrator based only on calibration validation data.

    This does not use home_id as a feature. It estimates one global scale factor
    to reduce systematic daily under/over-estimation. The balanced prediction
    remains the default; the calibrated one is optional.
    """
    y_true_sum = float(cal_eval[TARGET_COL].sum())
    y_pred_sum = float(cal_eval[pred_col].sum())

    raw_scale = (y_true_sum + 1e-6) / (y_pred_sum + 1e-6)
    raw_scale = float(np.clip(raw_scale, 0.90, 1.15))

    alpha = 0.50
    scale = float(1.0 + alpha * (raw_scale - 1.0))

    return {
        "type": "global_scale",
        "raw_scale": raw_scale,
        "alpha": alpha,
        "scale": scale,
        "default_prediction_column": "prediction_Wh",
        "optional_prediction_column": "prediction_daily_calibrated_Wh",
    }


def apply_global_daily_calibrator(pred: np.ndarray, calibrator: Dict[str, Any]) -> np.ndarray:
    pred = np.asarray(pred, dtype=np.float64)
    if calibrator.get("type") != "global_scale":
        return np.clip(pred, 0, None)
    return np.clip(pred * float(calibrator.get("scale", 1.0)), 0, None)


# ============================================================
# no-history BEHAVIOR PROFILES
# ============================================================

def build_behavior_profiles(train_df: pd.DataFrame) -> Dict[str, Any]:
    tmp = train_df.copy()

    static_dow_hour = (
        tmp
        .groupby(["static_profile_key", "day_of_week", "hour"])
        .agg(
            expected_static_dow_hour_Wh=(TARGET_COL, "mean"),
            count_static_dow_hour=(TARGET_COL, "size"),
        )
        .reset_index()
    )
    static_dow_hour.loc[
        static_dow_hour["count_static_dow_hour"] < MIN_STATIC_DOW_HOUR_COUNT,
        "expected_static_dow_hour_Wh",
    ] = np.nan

    static_hour = (
        tmp
        .groupby(["static_profile_key", "hour"])
        .agg(
            expected_static_hour_Wh=(TARGET_COL, "mean"),
            count_static_hour=(TARGET_COL, "size"),
        )
        .reset_index()
    )
    static_hour.loc[
        static_hour["count_static_hour"] < MIN_STATIC_HOUR_COUNT,
        "expected_static_hour_Wh",
    ] = np.nan

    static_overall = (
        tmp
        .groupby("static_profile_key")
        .agg(
            expected_static_overall_Wh=(TARGET_COL, "mean"),
            count_static_overall=(TARGET_COL, "size"),
        )
        .reset_index()
    )
    static_overall.loc[
        static_overall["count_static_overall"] < MIN_STATIC_OVERALL_COUNT,
        "expected_static_overall_Wh",
    ] = np.nan

    global_dow_hour = (
        tmp
        .groupby(["day_of_week", "hour"])
        .agg(expected_global_dow_hour_Wh=(TARGET_COL, "mean"))
        .reset_index()
    )

    global_hour = (
        tmp
        .groupby("hour")
        .agg(expected_global_hour_Wh=(TARGET_COL, "mean"))
        .reset_index()
    )

    global_overall = float(tmp[TARGET_COL].mean())

    return {
        "static_dow_hour": static_dow_hour,
        "static_hour": static_hour,
        "static_overall": static_overall,
        "global_dow_hour": global_dow_hour,
        "global_hour": global_hour,
        "global_overall": global_overall,
    }


def add_behavior_profile(df_part: pd.DataFrame, profiles: Dict[str, Any]) -> pd.DataFrame:
    out = df_part.copy()

    out = out.merge(
        profiles["static_dow_hour"],
        on=["static_profile_key", "day_of_week", "hour"],
        how="left",
    )

    out = out.merge(
        profiles["static_hour"],
        on=["static_profile_key", "hour"],
        how="left",
    )

    out = out.merge(
        profiles["static_overall"],
        on="static_profile_key",
        how="left",
    )

    out = out.merge(
        profiles["global_dow_hour"],
        on=["day_of_week", "hour"],
        how="left",
    )

    out = out.merge(
        profiles["global_hour"],
        on="hour",
        how="left",
    )

    fallback = [
        ("expected_static_dow_hour_Wh", "static_dow_hour"),
        ("expected_static_hour_Wh", "static_hour"),
        ("expected_static_overall_Wh", "static_overall"),
        ("expected_global_dow_hour_Wh", "global_dow_hour"),
        ("expected_global_hour_Wh", "global_hour"),
    ]

    out["expected_behavior_Wh"] = np.nan
    out["behavior_profile_source"] = "missing"

    for col, source in fallback:
        mask = out["expected_behavior_Wh"].isna() & out[col].notna()
        out.loc[mask, "expected_behavior_Wh"] = out.loc[mask, col]
        out.loc[mask, "behavior_profile_source"] = source

    out["expected_behavior_Wh"] = out["expected_behavior_Wh"].fillna(profiles["global_overall"])
    out.loc[out["behavior_profile_source"] == "missing", "behavior_profile_source"] = "global_overall"

    out["expected_behavior_Wh"] = out["expected_behavior_Wh"].clip(lower=0)
    out["expected_behavior_log1p"] = np.log1p(out["expected_behavior_Wh"])

    return out


# ============================================================
# no-history KNN PROFILES
# ============================================================

def build_home_static_table(train_df: pd.DataFrame) -> pd.DataFrame:
    cols = [
        HOME_COL,
        "residents",
        "total_floor_area_m2",
        "num_electric_appliances",
        "hometype",
        "urban_rural_class",
    ]

    available_cols = [c for c in cols if c in train_df.columns]
    home_static = train_df[available_cols].drop_duplicates(subset=[HOME_COL]).copy()

    for c in ["residents", "total_floor_area_m2", "num_electric_appliances"]:
        if c not in home_static.columns:
            home_static[c] = np.nan
        home_static[c] = pd.to_numeric(home_static[c], errors="coerce")

    for c in ["hometype", "urban_rural_class"]:
        if c not in home_static.columns:
            home_static[c] = "unknown"
        home_static[c] = home_static[c].fillna("unknown").astype(str).replace({"nan": "unknown", "None": "unknown"})

    return home_static


def fit_knn_static_model(train_df: pd.DataFrame, n_neighbors: int = 15) -> Dict[str, Any]:
    home_static = build_home_static_table(train_df)

    numeric_static = [
        "residents",
        "total_floor_area_m2",
        "num_electric_appliances",
    ]

    static_encoded = pd.get_dummies(
        home_static[numeric_static + ["hometype", "urban_rural_class"]],
        columns=["hometype", "urban_rural_class"],
        dummy_na=False,
    )

    medians = {}
    for c in numeric_static:
        med = static_encoded[c].median()
        if pd.isna(med):
            med = 0.0
        medians[c] = float(med)
        static_encoded[c] = static_encoded[c].fillna(med)

    static_encoded = static_encoded.fillna(0.0)

    scaler = StandardScaler()
    X_static = scaler.fit_transform(static_encoded.values)

    n_neighbors = min(n_neighbors, len(home_static))

    knn = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric="euclidean",
    )
    knn.fit(X_static)

    tmp = train_df.copy()

    home_dow_hour_profile = (
        tmp
        .groupby([HOME_COL, "day_of_week", "hour"], as_index=False)
        .agg(knn_source_dow_hour_Wh=(TARGET_COL, "mean"))
    )

    home_hour_profile = (
        tmp
        .groupby([HOME_COL, "hour"], as_index=False)
        .agg(knn_source_hour_Wh=(TARGET_COL, "mean"))
    )

    home_overall_profile = (
        tmp
        .groupby(HOME_COL, as_index=False)
        .agg(knn_source_overall_Wh=(TARGET_COL, "mean"))
    )

    return {
        "home_static": home_static,
        "static_columns": list(static_encoded.columns),
        "numeric_static": numeric_static,
        "medians": medians,
        "scaler": scaler,
        "knn": knn,
        "home_dow_hour_profile": home_dow_hour_profile,
        "home_hour_profile": home_hour_profile,
        "home_overall_profile": home_overall_profile,
        "n_neighbors": n_neighbors,
    }


def transform_home_static_for_knn(df_part: pd.DataFrame, knn_bundle: Dict[str, Any]):
    numeric_static = knn_bundle["numeric_static"]

    needed = [
        HOME_COL,
        "residents",
        "total_floor_area_m2",
        "num_electric_appliances",
        "hometype",
        "urban_rural_class",
    ]

    tmp = df_part[needed].drop_duplicates(subset=[HOME_COL]).copy()

    for c in numeric_static:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
        tmp[c] = tmp[c].fillna(knn_bundle["medians"].get(c, 0.0))

    for c in ["hometype", "urban_rural_class"]:
        tmp[c] = tmp[c].fillna("unknown").astype(str).replace({"nan": "unknown", "None": "unknown"})

    encoded = pd.get_dummies(
        tmp[numeric_static + ["hometype", "urban_rural_class"]],
        columns=["hometype", "urban_rural_class"],
        dummy_na=False,
    )

    encoded = encoded.reindex(
        columns=knn_bundle["static_columns"],
        fill_value=0.0,
    )

    X = knn_bundle["scaler"].transform(encoded.values)

    return tmp[[HOME_COL]].copy(), X


def add_knn_behavior_profile(df_part: pd.DataFrame, knn_bundle: Dict[str, Any]) -> pd.DataFrame:
    out = df_part.copy()

    homes_df, X = transform_home_static_for_knn(out, knn_bundle)
    _, indices = knn_bundle["knn"].kneighbors(X)

    train_homes = knn_bundle["home_static"][HOME_COL].values
    home_to_neighbors = {}

    for i, home_id in enumerate(homes_df[HOME_COL].values):
        neighbors = train_homes[indices[i]].tolist()

        if REMOVE_SAME_HOME_FROM_KNN:
            neighbors = [h for h in neighbors if h != home_id]

        if len(neighbors) == 0:
            neighbors = train_homes[indices[i]].tolist()

        home_to_neighbors[home_id] = neighbors

    rows = []
    for home_id, neighs in home_to_neighbors.items():
        for neigh in neighs:
            rows.append({
                HOME_COL: home_id,
                "neighbor_home_id": neigh,
            })

    neighbor_map = pd.DataFrame(rows)

    dow_hour = knn_bundle["home_dow_hour_profile"].rename(columns={HOME_COL: "neighbor_home_id"})
    tmp_dow_hour = neighbor_map.merge(dow_hour, on="neighbor_home_id", how="left")
    knn_dow_hour = (
        tmp_dow_hour
        .groupby([HOME_COL, "day_of_week", "hour"], as_index=False)
        .agg(knn_expected_dow_hour_Wh=("knn_source_dow_hour_Wh", "mean"))
    )

    hour_prof = knn_bundle["home_hour_profile"].rename(columns={HOME_COL: "neighbor_home_id"})
    tmp_hour = neighbor_map.merge(hour_prof, on="neighbor_home_id", how="left")
    knn_hour = (
        tmp_hour
        .groupby([HOME_COL, "hour"], as_index=False)
        .agg(knn_expected_hour_Wh=("knn_source_hour_Wh", "mean"))
    )

    overall_prof = knn_bundle["home_overall_profile"].rename(columns={HOME_COL: "neighbor_home_id"})
    tmp_overall = neighbor_map.merge(overall_prof, on="neighbor_home_id", how="left")
    knn_overall = (
        tmp_overall
        .groupby(HOME_COL, as_index=False)
        .agg(knn_expected_overall_Wh=("knn_source_overall_Wh", "mean"))
    )

    out = out.merge(knn_dow_hour, on=[HOME_COL, "day_of_week", "hour"], how="left")
    out = out.merge(knn_hour, on=[HOME_COL, "hour"], how="left")
    out = out.merge(knn_overall, on=HOME_COL, how="left")

    out["knn_expected_Wh"] = out["knn_expected_dow_hour_Wh"]
    out["knn_expected_Wh"] = out["knn_expected_Wh"].fillna(out["knn_expected_hour_Wh"])
    out["knn_expected_Wh"] = out["knn_expected_Wh"].fillna(out["knn_expected_overall_Wh"])
    out["knn_expected_Wh"] = out["knn_expected_Wh"].fillna(out["expected_behavior_Wh"])

    out["knn_expected_Wh"] = out["knn_expected_Wh"].clip(lower=0)
    out["knn_expected_log1p"] = np.log1p(out["knn_expected_Wh"])
    out["knn_minus_behavior_Wh"] = out["knn_expected_Wh"] - out["expected_behavior_Wh"]

    return out


# ============================================================
# DATA LOADING
# ============================================================

def load_and_prepare_base_data() -> Tuple[pd.DataFrame, list]:
    print("=" * 100)
    print("Loading dataset")
    print("=" * 100)

    df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL], low_memory=False)
    df = df.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)

    print(f"Rows loaded: {len(df):,}")
    print(f"Homes loaded: {df[HOME_COL].nunique()}")
    print(f"Date range: {df[TIME_COL].min()} to {df[TIME_COL].max()}")

    existing_bad = [h for h in BAD_HOMES if h in set(df[HOME_COL].unique())]

    if existing_bad:
        print("Removing bad homes:", existing_bad)
        df = df[~df[HOME_COL].isin(existing_bad)].copy()

    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[TARGET_COL]).copy()
    df[TARGET_COL] = df[TARGET_COL].clip(lower=0)

    for col in STATIC_NUMERIC:
        if col not in df.columns:
            df[col] = np.nan

    for col in BASE_CATEGORICAL:
        if col not in df.columns:
            df[col] = "unknown"

    df = add_time_features(df)
    df = add_static_bins(df)

    for c in BASE_CATEGORICAL:
        df[c] = df[c].fillna("unknown").astype(str).replace({"nan": "unknown", "None": "unknown"})

    for c in STATIC_NUMERIC + TIME_NUMERIC:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    print(f"Rows after cleaning: {len(df):,}")
    print(f"Homes after cleaning: {df[HOME_COL].nunique()}")

    return df, existing_bad


# ============================================================
# TRAIN WITH-HISTORY GENERIC MODEL
# ============================================================

def train_with_history_generic_model(df_base: pd.DataFrame) -> Dict[str, Any]:
    print("=" * 100)
    print("WITH-HISTORY TRAINING - GENERIC UI LGBM WITHOUT home_id")
    print("=" * 100)

    df_hist = add_lag_and_rolling_features(df_base)
    df_hist = df_hist.dropna(subset=REQUIRED_HISTORY_COLS).copy()
    df_hist = df_hist.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)

    print(f"Rows after history features: {len(df_hist):,}")
    print(f"Homes after history features: {df_hist[HOME_COL].nunique()}")
    print(f"Required minimum history hours: {MIN_REQUIRED_HISTORY_HOURS}")

    hist_train, hist_es, hist_cal, hist_test = temporal_split(df_hist)

    print("=" * 100)
    print("With-history generic temporal split")
    print("=" * 100)
    print(f"Train: {hist_train[TIME_COL].min()} to {hist_train[TIME_COL].max()} | rows: {len(hist_train):,}")
    print(f"Early-stop val: {hist_es[TIME_COL].min()} to {hist_es[TIME_COL].max()} | rows: {len(hist_es):,}")
    print(f"Calibration val: {hist_cal[TIME_COL].min()} to {hist_cal[TIME_COL].max()} | rows: {len(hist_cal):,}")
    print(f"Test: {hist_test[TIME_COL].min()} to {hist_test[TIME_COL].max()} | rows: {len(hist_test):,}")

    lag_cols = [c for c in df_hist.columns if c.startswith("lag_")]
    rolling_cols = [c for c in df_hist.columns if c.startswith("roll_")]

    numeric_cols = [
        c for c in STATIC_NUMERIC + TIME_NUMERIC + lag_cols + rolling_cols
        if c in hist_train.columns
    ]

    categorical_cols = [
        c for c in BASE_CATEGORICAL
        if c in hist_train.columns
    ]

    # Hard safety check: home_id must not be part of model inputs.
    forbidden_cols = {HOME_COL, "consumption_regime"}
    forbidden_in_features = forbidden_cols.intersection(set(numeric_cols + categorical_cols))
    if forbidden_in_features:
        raise ValueError(f"Forbidden generic UI features found: {forbidden_in_features}")

    for split_df in [hist_train, hist_es, hist_cal, hist_test]:
        for c in numeric_cols:
            split_df[c] = pd.to_numeric(split_df[c], errors="coerce")

        for c in categorical_cols:
            split_df[c] = split_df[c].fillna("unknown").astype(str).replace({"nan": "unknown", "None": "unknown"})

    print("=" * 100)
    print("With-history generic features")
    print("=" * 100)
    print(f"Numeric features: {len(numeric_cols)}")
    print(f"Categorical features: {len(categorical_cols)}")
    print("Categorical:", categorical_cols)
    print("Uses home_id as feature: False")
    print("Uses home-specific statistics: False")

    preprocessor = build_preprocessor(numeric_cols, categorical_cols)

    X_train = preprocessor.fit_transform(hist_train[numeric_cols + categorical_cols])
    X_es = preprocessor.transform(hist_es[numeric_cols + categorical_cols])
    X_cal = preprocessor.transform(hist_cal[numeric_cols + categorical_cols])
    X_test = preprocessor.transform(hist_test[numeric_cols + categorical_cols])

    y_train_raw = hist_train[TARGET_COL].values
    y_es_raw = hist_es[TARGET_COL].values

    if CLIP_TARGET_TRAIN:
        lo = np.quantile(y_train_raw, 0.005)
        hi = np.quantile(y_train_raw, 0.995)
        y_train_used = np.clip(y_train_raw, lo, hi)
        clip_bounds = {"low": float(lo), "high": float(hi)}
    else:
        y_train_used = y_train_raw
        clip_bounds = None

    y_train = transform_target(y_train_used)
    y_es = transform_target(y_es_raw)

    print("=" * 100)
    print("With-history generic matrices")
    print("=" * 100)
    print(f"X_train: {X_train.shape}")
    print(f"X_es:    {X_es.shape}")
    print(f"X_cal:   {X_cal.shape}")
    print(f"X_test:  {X_test.shape}")

    model = lgb.LGBMRegressor(**HISTORY_LGBM_PARAMS)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_es, y_es)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(stopping_rounds=150),
            lgb.log_evaluation(period=200),
        ],
    )

    best_iteration = model.best_iteration_

    pred_cal = inverse_target(model.predict(X_cal, num_iteration=best_iteration))
    pred_test = inverse_target(model.predict(X_test, num_iteration=best_iteration))

    pred_cal = np.clip(pred_cal, 0, None)
    pred_test = np.clip(pred_test, 0, None)

    cal_eval = hist_cal[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    cal_eval["prediction_Wh"] = pred_cal

    global_daily_calibrator = fit_global_daily_calibrator(cal_eval, "prediction_Wh")
    save_json(global_daily_calibrator, WITH_HISTORY_DIR / "global_daily_calibrator.json")

    test_eval = hist_test[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    test_eval["prediction_Wh"] = pred_test
    test_eval["prediction_daily_calibrated_Wh"] = apply_global_daily_calibrator(
        pred_test,
        global_daily_calibrator,
    )

    metrics_balanced = evaluate_predictions(test_eval, "prediction_Wh")
    metrics_daily_calibrated = evaluate_predictions(test_eval, "prediction_daily_calibrated_Wh")

    print_metrics("FINAL TEST METRICS - WITH HISTORY GENERIC BALANCED", metrics_balanced)
    print_metrics("FINAL TEST METRICS - WITH HISTORY GENERIC DAILY-CALIBRATED OPTIONAL", metrics_daily_calibrated)

    model_path = WITH_HISTORY_DIR / "model.joblib"
    preprocessor_path = WITH_HISTORY_DIR / "preprocessor.pkl"
    feature_config_path = WITH_HISTORY_DIR / "feature_config.json"
    metadata_path = WITH_HISTORY_DIR / "metadata.json"
    predictions_path = WITH_HISTORY_DIR / "test_predictions.csv"

    joblib.dump(model, model_path)
    joblib.dump(preprocessor, preprocessor_path)

    feature_config = {
        "mode": "with_history_generic",
        "ui_generic_user_compatible": True,
        "requires_user_home_id": False,
        "uses_home_id_as_feature": False,
        "uses_home_stats": False,
        "uses_consumption_regime": False,
        "uses_lag_features": True,
        "uses_rolling_features": True,
        "uses_knn_profiles": False,
        "target_col": TARGET_COL,
        "time_col": TIME_COL,
        "home_col_internal_training_only": HOME_COL,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN,
        "clip_bounds": clip_bounds,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "static_numeric": STATIC_NUMERIC,
        "time_numeric": TIME_NUMERIC,
        "base_categorical": BASE_CATEGORICAL,
        "required_history_cols": REQUIRED_HISTORY_COLS,
        "min_required_history_hours": MIN_REQUIRED_HISTORY_HOURS,
        "lag_hours": LAG_HOURS,
        "rolling_windows": ROLLING_WINDOWS,
        "rolling_extreme_windows": ROLLING_EXTREME_WINDOWS,
        "default_prediction_column": "prediction_Wh",
        "optional_prediction_column": "prediction_daily_calibrated_Wh",
        "inference_note": (
            "For 24-hour UI forecasts, lag features must be built from the user-provided recent "
            "history. If future lag values are needed inside the forecast horizon, the API should "
            "use a recursive strategy or a clearly defined fallback."
        ),
    }
    save_json(feature_config, feature_config_path)

    metadata = {
        "model_family": "LightGBM",
        "model_name": "lgbm_withhistory_generic_no_home_id",
        "mode": "with_history_generic",
        "ui_role": "with_history_default_candidate_for_generic_users",
        "best_iteration": int(best_iteration),
        "lgbm_params": HISTORY_LGBM_PARAMS,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN,
        "clip_bounds": clip_bounds,
        "metrics_balanced": metrics_balanced,
        "metrics_daily_calibrated_optional": metrics_daily_calibrated,
        "global_daily_calibrator": global_daily_calibrator,
        "artifact_files": {
            "model": str(model_path),
            "preprocessor": str(preprocessor_path),
            "feature_config": str(feature_config_path),
            "metadata": str(metadata_path),
            "global_daily_calibrator": str(WITH_HISTORY_DIR / "global_daily_calibrator.json"),
            "predictions": str(predictions_path),
        },
        "train_period": {
            "start": str(hist_train[TIME_COL].min()),
            "end": str(hist_train[TIME_COL].max()),
            "rows": int(len(hist_train)),
        },
        "early_stop_validation_period": {
            "start": str(hist_es[TIME_COL].min()),
            "end": str(hist_es[TIME_COL].max()),
            "rows": int(len(hist_es)),
        },
        "calibration_validation_period": {
            "start": str(hist_cal[TIME_COL].min()),
            "end": str(hist_cal[TIME_COL].max()),
            "rows": int(len(hist_cal)),
        },
        "test_period": {
            "start": str(hist_test[TIME_COL].min()),
            "end": str(hist_test[TIME_COL].max()),
            "rows": int(len(hist_test)),
        },
    }
    save_json(metadata, metadata_path)
    test_eval.to_csv(predictions_path, index=False)

    return {
        "metrics_balanced": metrics_balanced,
        "metrics_daily_calibrated_optional": metrics_daily_calibrated,
        "model_path": str(model_path),
        "preprocessor_path": str(preprocessor_path),
        "feature_config_path": str(feature_config_path),
        "metadata_path": str(metadata_path),
        "predictions_path": str(predictions_path),
        "best_iteration": int(best_iteration),
    }


# ============================================================
# TRAIN no-history OPTIONAL MODEL
# ============================================================

def train_no_history_optional_model(df_base: pd.DataFrame) -> Dict[str, Any]:
    print("=" * 100)
    print("no-history TRAINING - OPTIONAL BEHAVIOR + KNN LGBM")
    print("=" * 100)

    df_NOHISTORY = df_base.sort_values(TIME_COL).reset_index(drop=True).copy()

    train_df, es_df, cal_df, test_df = temporal_split(df_NOHISTORY)

    print("=" * 100)
    print("no-history temporal split")
    print("=" * 100)
    print(f"Train: {train_df[TIME_COL].min()} to {train_df[TIME_COL].max()} | rows: {len(train_df):,}")
    print(f"Early-stop val: {es_df[TIME_COL].min()} to {es_df[TIME_COL].max()} | rows: {len(es_df):,}")
    print(f"Calibration val: {cal_df[TIME_COL].min()} to {cal_df[TIME_COL].max()} | rows: {len(cal_df):,}")
    print(f"Test: {test_df[TIME_COL].min()} to {test_df[TIME_COL].max()} | rows: {len(test_df):,}")

    print("=" * 100)
    print("Building train-only behavioral profiles")
    print("=" * 100)

    behavior_profiles = build_behavior_profiles(train_df)

    # Save both transparent files and a compact joblib bundle for API loading.
    joblib.dump(behavior_profiles, no_history_DIR / "behavior_profiles.joblib")

    for name, obj in behavior_profiles.items():
        if isinstance(obj, pd.DataFrame):
            obj.to_csv(no_history_DIR / f"behavior_profile_{name}.csv", index=False)
        else:
            save_json({"value": obj}, no_history_DIR / f"behavior_profile_{name}.json")

    train_df = add_behavior_profile(train_df, behavior_profiles)
    es_df = add_behavior_profile(es_df, behavior_profiles)
    cal_df = add_behavior_profile(cal_df, behavior_profiles)
    test_df = add_behavior_profile(test_df, behavior_profiles)

    print("Behavior profile source usage in test:")
    print(test_df["behavior_profile_source"].value_counts(dropna=False))

    print("=" * 100)
    print("Building train-only KNN similar-home profiles")
    print("=" * 100)

    knn_bundle = fit_knn_static_model(train_df, n_neighbors=N_KNN_NEIGHBORS)

    train_df = add_knn_behavior_profile(train_df, knn_bundle)
    es_df = add_knn_behavior_profile(es_df, knn_bundle)
    cal_df = add_knn_behavior_profile(cal_df, knn_bundle)
    test_df = add_knn_behavior_profile(test_df, knn_bundle)

    # Save compact full bundle for easiest API loading.
    joblib.dump(knn_bundle, no_history_DIR / "knn_bundle.joblib")

    # Save transparent component files as well.
    joblib.dump(knn_bundle["scaler"], no_history_DIR / "knn_static_scaler.pkl")
    joblib.dump(knn_bundle["knn"], no_history_DIR / "knn_model.pkl")
    knn_bundle["home_static"].to_csv(no_history_DIR / "knn_home_static.csv", index=False)
    knn_bundle["home_dow_hour_profile"].to_csv(no_history_DIR / "knn_home_dow_hour_profile.csv", index=False)
    knn_bundle["home_hour_profile"].to_csv(no_history_DIR / "knn_home_hour_profile.csv", index=False)
    knn_bundle["home_overall_profile"].to_csv(no_history_DIR / "knn_home_overall_profile.csv", index=False)

    save_json(
        {
            "static_columns": knn_bundle["static_columns"],
            "numeric_static": knn_bundle["numeric_static"],
            "medians": knn_bundle["medians"],
            "n_neighbors": knn_bundle["n_neighbors"],
            "remove_same_home_from_knn": REMOVE_SAME_HOME_FROM_KNN,
        },
        no_history_DIR / "knn_config.json",
    )

    numeric_cols = [
        c for c in STATIC_NUMERIC + TIME_NUMERIC + NOHISTORY_EXTRA_NUMERIC
        if c in train_df.columns
    ]

    categorical_cols = [
        c for c in NOHISTORY_CATEGORICAL
        if c in train_df.columns
    ]

    # Hard safety check: home_id must not be part of model inputs.
    forbidden_cols = {HOME_COL, "static_profile_key"}
    forbidden_in_features = forbidden_cols.intersection(set(numeric_cols + categorical_cols))
    if forbidden_in_features:
        raise ValueError(f"Forbidden no-history UI features found: {forbidden_in_features}")

    for split_df in [train_df, es_df, cal_df, test_df]:
        for c in numeric_cols:
            split_df[c] = pd.to_numeric(split_df[c], errors="coerce")

        for c in categorical_cols:
            split_df[c] = split_df[c].fillna("unknown").astype(str).replace({"nan": "unknown", "None": "unknown"})

    print("=" * 100)
    print("no-history optional features")
    print("=" * 100)
    print(f"Numeric features: {len(numeric_cols)}")
    print(f"Categorical features: {len(categorical_cols)}")
    print("Categorical:", categorical_cols)
    print("Uses home_id as feature: False")

    preprocessor = build_preprocessor(numeric_cols, categorical_cols)

    X_train = preprocessor.fit_transform(train_df[numeric_cols + categorical_cols])
    X_es = preprocessor.transform(es_df[numeric_cols + categorical_cols])
    X_cal = preprocessor.transform(cal_df[numeric_cols + categorical_cols])
    X_test = preprocessor.transform(test_df[numeric_cols + categorical_cols])

    y_train_raw = train_df[TARGET_COL].values
    y_es_raw = es_df[TARGET_COL].values

    if CLIP_TARGET_TRAIN:
        lo = np.quantile(y_train_raw, 0.005)
        hi = np.quantile(y_train_raw, 0.995)
        y_train_used = np.clip(y_train_raw, lo, hi)
        clip_bounds = {"low": float(lo), "high": float(hi)}
    else:
        y_train_used = y_train_raw
        clip_bounds = None

    y_train = transform_target(y_train_used)
    y_es = transform_target(y_es_raw)

    print("=" * 100)
    print("no-history optional matrices")
    print("=" * 100)
    print(f"X_train: {X_train.shape}")
    print(f"X_es:    {X_es.shape}")
    print(f"X_cal:   {X_cal.shape}")
    print(f"X_test:  {X_test.shape}")

    test_eval = test_df[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    test_eval["expected_behavior_Wh"] = test_df["expected_behavior_Wh"].values
    test_eval["knn_expected_Wh"] = test_df["knn_expected_Wh"].values

    metrics_behavior = evaluate_predictions(test_eval, "expected_behavior_Wh")
    metrics_knn = evaluate_predictions(test_eval, "knn_expected_Wh")

    print_metrics("FINAL TEST METRICS - NOHISTORY BEHAVIOR PROFILE ONLY", metrics_behavior)
    print_metrics("FINAL TEST METRICS - NOHISTORY KNN PROFILE ONLY", metrics_knn)

    model = lgb.LGBMRegressor(**NOHISTORY_LGBM_PARAMS)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_es, y_es)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(stopping_rounds=150),
            lgb.log_evaluation(period=200),
        ],
    )

    best_iteration = model.best_iteration_

    pred_cal = inverse_target(model.predict(X_cal, num_iteration=best_iteration))
    pred_test = inverse_target(model.predict(X_test, num_iteration=best_iteration))

    pred_cal = np.clip(pred_cal, 0, None)
    pred_test = np.clip(pred_test, 0, None)

    cal_eval = cal_df[[HOME_COL, TIME_COL, TARGET_COL]].copy()
    cal_eval["prediction_Wh"] = pred_cal
    cal_eval["expected_behavior_Wh"] = cal_df["expected_behavior_Wh"].values
    cal_eval["knn_expected_Wh"] = cal_df["knn_expected_Wh"].values

    test_eval["prediction_Wh"] = pred_test

    metrics_cal = evaluate_predictions(cal_eval, "prediction_Wh")
    metrics_test = evaluate_predictions(test_eval, "prediction_Wh")

    print_metrics("CALIBRATION VALIDATION METRICS - no-history OPTIONAL LGBM", metrics_cal)
    print_metrics("FINAL TEST METRICS - no-history OPTIONAL LGBM", metrics_test)

    comparison_df = pd.DataFrame([
        {"method": "behavior_profile_only", **metrics_behavior},
        {"method": "knn_profile_only", **metrics_knn},
        {"method": "NOHISTORYstart_lgbm_behavior_knn_optional", **metrics_test},
    ])
    comparison_df.to_csv(no_history_DIR / "NOHISTORYstart_test_comparison.csv", index=False)

    source_rows = []
    for source, g in test_df.assign(prediction_Wh=pred_test).groupby("behavior_profile_source"):
        eval_g = g[[HOME_COL, TIME_COL, TARGET_COL, "prediction_Wh"]].copy()
        m = evaluate_predictions(eval_g, "prediction_Wh")
        source_rows.append({
            "behavior_profile_source": source,
            "rows": int(len(g)),
            "homes": int(g[HOME_COL].nunique()),
            **m,
        })

    source_metrics_df = pd.DataFrame(source_rows).sort_values("RMSE", ascending=False)
    source_metrics_df.to_csv(no_history_DIR / "metrics_by_behavior_source.csv", index=False)

    home_rows = []
    test_eval_full = test_eval.copy()
    test_eval_full["behavior_profile_source"] = test_df["behavior_profile_source"].values

    for home_id, g in test_eval_full.groupby(HOME_COL):
        m = evaluate_predictions(g, "prediction_Wh")
        home_rows.append({
            HOME_COL: home_id,
            "rows": int(len(g)),
            "behavior_profile_source_mode": (
                g["behavior_profile_source"].mode().iloc[0]
                if len(g["behavior_profile_source"].mode()) else "unknown"
            ),
            **m,
        })

    home_metrics_df = pd.DataFrame(home_rows).sort_values("RMSE", ascending=False)
    home_metrics_df.to_csv(no_history_DIR / "metrics_by_home.csv", index=False)

    model_path = no_history_DIR / "model.joblib"
    preprocessor_path = no_history_DIR / "preprocessor.pkl"
    feature_config_path = no_history_DIR / "feature_config.json"
    metadata_path = no_history_DIR / "metadata.json"
    predictions_path = no_history_DIR / "test_predictions.csv"

    joblib.dump(model, model_path)
    joblib.dump(preprocessor, preprocessor_path)

    feature_config = {
        "mode": "no_history_optional",
        "ui_generic_user_compatible": True,
        "requires_user_home_id": False,
        "uses_home_id_as_feature": False,
        "uses_lag_features": False,
        "uses_rolling_features": False,
        "uses_behavior_profiles": True,
        "uses_knn_profiles": True,
        "target_col": TARGET_COL,
        "time_col": TIME_COL,
        "home_col_internal_only_for_knn": HOME_COL,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN,
        "clip_bounds": clip_bounds,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "static_numeric": STATIC_NUMERIC,
        "time_numeric": TIME_NUMERIC,
        "NOHISTORY_extra_numeric": NOHISTORY_EXTRA_NUMERIC,
        "NOHISTORY_categorical": NOHISTORY_CATEGORICAL,
        "requires_internal_pseudo_home_id_for_api_knn": True,
        "suggested_api_pseudo_home_id": "ui_user",
        "default_prediction_column": "prediction_Wh",
        "role_note": (
            "This LightGBM no-history model is API-compatible and can be used for comparison. "
            "The final project default no-history model may still be the Random Forest final refit model."
        ),
    }
    save_json(feature_config, feature_config_path)

    metadata = {
        "model_family": "LightGBM",
        "model_name": "lgbm_NOHISTORYstart_behavior_knn_optional",
        "mode": "no_history_optional",
        "ui_role": "optional_comparison_model",
        "best_iteration": int(best_iteration),
        "lgbm_params": NOHISTORY_LGBM_PARAMS,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN,
        "clip_bounds": clip_bounds,
        "metrics_behavior_profile_only": metrics_behavior,
        "metrics_knn_profile_only": metrics_knn,
        "metrics_NOHISTORYstart_lgbm": metrics_test,
        "calibration_validation_metrics": metrics_cal,
        "artifact_files": {
            "model": str(model_path),
            "preprocessor": str(preprocessor_path),
            "feature_config": str(feature_config_path),
            "metadata": str(metadata_path),
            "behavior_profiles_joblib": str(no_history_DIR / "behavior_profiles.joblib"),
            "knn_bundle_joblib": str(no_history_DIR / "knn_bundle.joblib"),
            "knn_config": str(no_history_DIR / "knn_config.json"),
            "knn_model": str(no_history_DIR / "knn_model.pkl"),
            "knn_scaler": str(no_history_DIR / "knn_static_scaler.pkl"),
            "behavior_static_dow_hour": str(no_history_DIR / "behavior_profile_static_dow_hour.csv"),
            "behavior_static_hour": str(no_history_DIR / "behavior_profile_static_hour.csv"),
            "behavior_static_overall": str(no_history_DIR / "behavior_profile_static_overall.csv"),
            "behavior_global_dow_hour": str(no_history_DIR / "behavior_profile_global_dow_hour.csv"),
            "behavior_global_hour": str(no_history_DIR / "behavior_profile_global_hour.csv"),
            "behavior_global_overall": str(no_history_DIR / "behavior_profile_global_overall.json"),
            "predictions": str(predictions_path),
        },
        "train_period": {
            "start": str(train_df[TIME_COL].min()),
            "end": str(train_df[TIME_COL].max()),
            "rows": int(len(train_df)),
        },
        "early_stop_validation_period": {
            "start": str(es_df[TIME_COL].min()),
            "end": str(es_df[TIME_COL].max()),
            "rows": int(len(es_df)),
        },
        "calibration_validation_period": {
            "start": str(cal_df[TIME_COL].min()),
            "end": str(cal_df[TIME_COL].max()),
            "rows": int(len(cal_df)),
        },
        "test_period": {
            "start": str(test_df[TIME_COL].min()),
            "end": str(test_df[TIME_COL].max()),
            "rows": int(len(test_df)),
        },
    }
    save_json(metadata, metadata_path)
    test_eval.to_csv(predictions_path, index=False)

    return {
        "metrics_behavior_profile_only": metrics_behavior,
        "metrics_knn_profile_only": metrics_knn,
        "metrics_NOHISTORYstart_lgbm": metrics_test,
        "calibration_validation_metrics": metrics_cal,
        "model_path": str(model_path),
        "preprocessor_path": str(preprocessor_path),
        "feature_config_path": str(feature_config_path),
        "metadata_path": str(metadata_path),
        "predictions_path": str(predictions_path),
        "best_iteration": int(best_iteration),
    }


# ============================================================
# MAIN
# ============================================================

def main():
    df_base, existing_bad = load_and_prepare_base_data()

    summary = {
        "script": "train_lgbm_api_generic_models.py",
        "data_path": str(DATA_PATH),
        "target": TARGET_COL,
        "time_col": TIME_COL,
        "home_col": HOME_COL,
        "test_days": TEST_DAYS,
        "val_days_total": VAL_DAYS_TOTAL,
        "cal_days": CAL_DAYS,
        "bad_homes_removed": existing_bad,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN,
        "output_root": str(OUT_ROOT),
        "model_selection": {
            "with_history_generic_lgbm_default_candidate": "with_history_generic/model.joblib",
            "no_history_lgbm_optional": "no_history_optional/model.joblib",
            "project_default_note": (
                "For the final UI, use this generic LGBM with-history model when the user supplies "
                "recent consumption history but does not supply home_id. For no-history, the final "
                "project default can remain the Random Forest simple final refit model; the LGBM "
                "no-history model trained here is kept as an optional comparison model."
            ),
            "routing_rule": (
                "Use with_history_generic when recent consumption history is available. "
                "Use no_history_optional only for comparison or if explicitly selected. "
                "Do not route generic UI users to known-home models that require home_id."
            ),
        },
    }

    if TRAIN_WITH_HISTORY_GENERIC:
        with_history_result = train_with_history_generic_model(df_base)
        summary["with_history_generic"] = with_history_result
    else:
        summary["with_history_generic"] = None

    if TRAIN_no_history_OPTIONAL:
        no_history_result = train_no_history_optional_model(df_base)
        summary["no_history_optional"] = no_history_result
    else:
        summary["no_history_optional"] = None

    save_json(summary, OUT_ROOT / "training_summary.json")

    print("=" * 100)
    print("FINAL GENERIC UI LGBM API MODELS TRAINING COMPLETED")
    print("=" * 100)
    print("OUT_ROOT:", OUT_ROOT)
    print("With-history generic folder:", WITH_HISTORY_DIR)
    print("no-history optional folder:", no_history_DIR)
    print("Training summary:", OUT_ROOT / "training_summary.json")

    if summary.get("with_history_generic"):
        print("With-history generic model:", summary["with_history_generic"]["model_path"])

    if summary.get("no_history_optional"):
        print("no-history optional model:", summary["no_history_optional"]["model_path"])


if __name__ == "__main__":
    main()


Loading dataset
Rows loaded: 1,529,350
Homes loaded: 254
Date range: 2016-08-10 10:00:00 to 2018-06-30 23:00:00
Removing bad homes: ['home171', 'home219', 'home228', 'home271']
Rows after cleaning: 1,513,767
Homes after cleaning: 250
WITH-HISTORY TRAINING - GENERIC UI LGBM WITHOUT home_id
Rows after history features: 1,471,767
Homes after history features: 250
Required minimum history hours: 168
With-history generic temporal split
Train: 2016-08-17 10:00:00 to 2018-05-01 23:00:00 | rows: 1,203,425
Early-stop val: 2018-05-02 00:00:00 to 2018-05-16 23:00:00 | rows: 66,624
Calibration val: 2018-05-17 00:00:00 to 2018-05-31 23:00:00 | rows: 72,482
Test: 2018-06-01 00:00:00 to 2018-06-30 23:00:00 | rows: 129,236
With-history generic features
Numeric features: 47
Categorical features: 2
Categorical: ['hometype', 'urban_rural_class']
Uses home_id as feature: False
Uses home-specific statistics: False
With-history generic matrices
X_train: (1203425, 52)
X_es:    (66624, 52)
X_cal:   (72482, 52